# NutriMatch 2-Year Overweight/Obesity Prediction With Enhancement Arms

This notebook focuses only on the NutriMatch paper's Figure 3c-style analysis: predicting overweight/obesity status at about 2-year follow-up from diet-derived features.

It compares:

- Age + sex
- Age + sex + paper-basic nutrients
- Age + sex + NutriMatch all nutrients
- Age + sex + Diet Data Enhancement feature sets

The model uses 5-fold stratified cross-validation and reports AUROC, AUPRC, accuracy, balanced accuracy, and F1. Dietary feature tables are cached under this notebook's own task folder, using the paper-style filter of diet-logging days with at least 800 kcal when calorie columns are available.

In [ ]:
from pathlib import Path
import gc
import json
import os
import re
import sys
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except Exception:
    LGBMClassifier = None
    LIGHTGBM_AVAILABLE = False

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

RUN_TRAINING = os.environ.get('DDE_RUN_TRAINING', '0') == '1'
RANDOM_STATE = int(os.environ.get('DDE_RANDOM_STATE', '42'))
N_SPLITS = int(os.environ.get('DDE_N_SPLITS', '5'))
MIN_DAILY_KCAL = float(os.environ.get('DDE_MIN_DAILY_KCAL', '800'))
FOLLOWUP_TARGET_DAYS = int(os.environ.get('DDE_FOLLOWUP_TARGET_DAYS', '730'))
FOLLOWUP_MIN_DAYS = int(os.environ.get('DDE_FOLLOWUP_MIN_DAYS', '540'))
FOLLOWUP_MAX_DAYS = int(os.environ.get('DDE_FOLLOWUP_MAX_DAYS', '900'))
BMI_CUTOFF = float(os.environ.get('DDE_BMI_OVERWEIGHT_CUTOFF', '25'))
MODEL_NAME = os.environ.get('DDE_MODEL', 'lightgbm')
ALLOW_MODEL_FALLBACK = os.environ.get('DDE_ALLOW_MODEL_FALLBACK', '1') == '1'
RUN_RF_SUPPLEMENT = os.environ.get('DDE_RUN_RF_SUPPLEMENT', '1') == '1'
X_BUILD_BATCH_SIZE = int(os.environ.get('DDE_X_BATCH_SIZE', '20'))
FEATURE_SET_FILTER = [x.strip() for x in os.environ.get('DDE_FEATURE_SET_FILTER', '').split(',') if x.strip()]

print('RUN_TRAINING:', RUN_TRAINING)
print('LightGBM available:', LIGHTGBM_AVAILABLE)
print('MODEL_NAME:', MODEL_NAME)
print('Allow model fallback:', ALLOW_MODEL_FALLBACK)
print('Run random forest supplement:', RUN_RF_SUPPLEMENT)
print('Minimum kcal/day filter:', MIN_DAILY_KCAL)
print('X build batch size:', X_BUILD_BATCH_SIZE)
print('Feature set filter:', FEATURE_SET_FILTER or 'all')
print('Follow-up window:', FOLLOWUP_MIN_DAYS, 'to', FOLLOWUP_MAX_DAYS, 'days; target=', FOLLOWUP_TARGET_DAYS)

## Paths And Feature Arms

In [ ]:
PROJECT_ROOT = Path.cwd()
tre_root = Path('/home/ec2-user/studies/Diet_Data_Enhancement_Project/Diet_Data_Enhancement_TRE')
if not (PROJECT_ROOT / 'downstream_analysis').exists() and (tre_root / 'downstream_analysis').exists():
    PROJECT_ROOT = tre_root
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_STEM = 'nutrimatch_two_year_obesity_with_enhancements'
TASK_DIR = PROJECT_ROOT / 'downstream_analysis/tasks' / NOTEBOOK_STEM
OUT_DIR = TASK_DIR / 'outputs'
FIG_DIR = OUT_DIR / 'figures'
CACHE_DIR = OUT_DIR / 'cache'
LOG_DIR = OUT_DIR / 'logs'
TRE_INPUTS = PROJECT_ROOT / 'tre_inputs'
for d in [TASK_DIR, OUT_DIR, FIG_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

FEATURE_SETS = [
    {'name': 'basic_nutrimatch', 'label': 'NutriMatch all nutrients', 'path': 'outputs/enhanced_hpp/2.nutrimatch_based/hpp_feature_matrix_per_100g.csv', 'feature_mode': 'enriched'},
    {'name': 'denovo_microbiome', 'label': 'De novo microbiome-oriented', 'path': 'outputs/downstream_features/denovo/microbiome/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'denovo_cardiometabolic', 'label': 'De novo cardiometabolic', 'path': 'outputs/downstream_features/denovo/cardiometabolic/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_microbiome', 'label': 'NutriMatch microbiome-oriented', 'path': 'outputs/downstream_features/nutrimatch_based/microbiome/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_cardiometabolic', 'label': 'NutriMatch cardiometabolic', 'path': 'outputs/downstream_features/nutrimatch_based/cardiometabolic/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_broad_diet_health', 'label': 'NutriMatch broad diet-health', 'path': 'outputs/downstream_features/nutrimatch_based/broad_diet_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'nutrimatch_mental_health', 'label': 'NutriMatch mental-health', 'path': 'outputs/downstream_features/nutrimatch_based/mental_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'denovo_broad_diet_health', 'label': 'De novo broad diet-health', 'path': 'outputs/downstream_features/denovo/broad_diet_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
    {'name': 'denovo_mental_health', 'label': 'De novo mental-health', 'path': 'outputs/downstream_features/denovo/mental_health/hpp_downstream_feature_table.csv', 'feature_mode': 'kg'},
]
FEATURE_SETS = [fs for fs in FEATURE_SETS if (PROJECT_ROOT / fs['path']).exists()]
if FEATURE_SET_FILTER:
    wanted = set(FEATURE_SET_FILTER) | {'basic_nutrimatch'}
    FEATURE_SETS = [fs for fs in FEATURE_SETS if fs['name'] in wanted]

ARM_LABELS = {
    'age_sex_only': 'Age + sex',
    'paper_basic_nutrients': 'Age + sex + basic nutrients',
    'nutrimatch_all': 'Age + sex + NutriMatch all nutrients',
}
for fs in FEATURE_SETS:
    if fs['name'] != 'basic_nutrimatch':
        ARM_LABELS[fs['name']] = 'Age + sex + ' + fs['label']

ARM_ORDER = ['age_sex_only', 'paper_basic_nutrients', 'nutrimatch_all'] + [fs['name'] for fs in FEATURE_SETS if fs['name'] != 'basic_nutrimatch']

print('Project root:', PROJECT_ROOT)
print('Output directory:', OUT_DIR)
print('Feature sets found:', [fs['name'] for fs in FEATURE_SETS])

## Console Runner Status

Long training should be launched by the background runner. This cell only inspects the newest temporary log.

In [ ]:
def newest_tmp_log(prefix='nutrimatch_two_year_obesity_with_enhancements'):
    logs = sorted(Path('/tmp').glob(prefix + '*.log'), key=lambda p: p.stat().st_mtime, reverse=True)
    return logs[0] if logs else None

latest_pid = LOG_DIR / 'background_training_latest.pid'
print('PID file exists:', latest_pid.exists())
if latest_pid.exists():
    print('PID:', latest_pid.read_text().strip())
log = newest_tmp_log()
print('Newest /tmp log:', log)
if log and log.exists():
    print(log.read_text(errors='replace')[-4000:])

## TRE Loading Helpers

In [ ]:
from downstream_analysis.data_handelling.pheno_loader_export import (
    make_loader,
    load_table_from_loader,
    dataframe_with_index_columns,
)


def read_any(path):
    path = Path(path)
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    return pd.read_csv(path, low_memory=False)


def write_any(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == '.parquet':
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False)
    return path


def clean_feature_name(name):
    text = str(name).lower()
    text = re.sub(r'^(enriched_|kg_|food_card_)', '', text)
    return re.sub(r'[^a-z0-9]+', '_', text).strip('_')


def normalize_pid_series(s):
    return s.astype(str)


def find_participant_col(df):
    for col in ['participant_id', 'Participant_Study_ID', 'research_stage_id', 'user_id', 'RegistrationCode']:
        if col in df.columns:
            return col
    for col in df.columns:
        text = str(col).lower()
        if 'participant' in text or 'research_stage' in text:
            return col
    return None


def try_load_pheno_table(dataset, table=None):
    try:
        loader = make_loader(dataset, age_sex_dataset=None, errors='warn')
        table_name = table or dataset
        try:
            df = load_table_from_loader(loader, dataset, table_name, required=False)
        except Exception:
            df = None
        if df is None:
            dfs = getattr(loader, 'dfs', {})
            if table_name in dfs:
                df = dataframe_with_index_columns(dfs[table_name])
            elif len(dfs) == 1:
                df = dataframe_with_index_columns(next(iter(dfs.values())))
        return df, loader
    except Exception as exc:
        print(f'Could not load {dataset}/{table or dataset}: {exc}')
        return None, None


def find_first_matching_column(df, patterns):
    for pat in patterns:
        rx = re.compile(pat, re.IGNORECASE)
        exact = [c for c in df.columns if rx.fullmatch(str(c))]
        if exact:
            return exact[0]
        partial = [c for c in df.columns if rx.search(str(c))]
        if partial:
            return partial[0]
    return None

## Build 2-Year Overweight/Obesity Target

This follows the paper description as closely as possible from TRE tables: baseline BMI is the first anthropometrics BMI measurement, follow-up BMI is the measurement closest to 730 days within an 18-30 month window, and overweight/obesity is BMI >= 25 at follow-up.

In [ ]:
def build_two_year_obesity_target():
    cache = CACHE_DIR / 'two_year_overweight_obesity_target.csv'
    if cache.exists():
        target = pd.read_csv(cache, low_memory=False)
        target['participant_id'] = target['participant_id'].astype(str)
        print('Loaded cached target:', cache, target.shape)
        return target

    frame, _ = try_load_pheno_table('anthropometrics', 'anthropometrics')
    if frame is None or frame.empty:
        raise FileNotFoundError('Could not load anthropometrics/anthropometrics from PhenoLoader.')

    pid_col = find_participant_col(frame)
    bmi_col = find_first_matching_column(frame, [r'bmi', r'body.*mass.*index'])
    date_col = find_first_matching_column(frame, [r'collection_timestamp', r'collection_date', r'date', r'timestamp'])
    if pid_col is None or bmi_col is None or date_col is None:
        print('Anthropometrics columns:', list(frame.columns))
        raise ValueError('Need participant, BMI, and date columns to derive 2-year obesity target.')

    tmp = frame[[pid_col, bmi_col, date_col]].copy().rename(columns={pid_col: 'participant_id', bmi_col: 'bmi', date_col: 'date'})
    tmp['participant_id'] = normalize_pid_series(tmp['participant_id'])
    tmp['bmi'] = pd.to_numeric(tmp['bmi'], errors='coerce')
    tmp['date'] = pd.to_datetime(tmp['date'], errors='coerce')
    tmp = tmp.dropna(subset=['participant_id', 'bmi', 'date']).sort_values(['participant_id', 'date'])

    baseline = tmp.groupby('participant_id', as_index=False).first().rename(columns={'date': 'baseline_date', 'bmi': 'baseline_bmi'})
    joined = tmp.merge(baseline[['participant_id', 'baseline_date', 'baseline_bmi']], on='participant_id', how='inner')
    joined['days_from_baseline'] = (joined['date'] - joined['baseline_date']).dt.days
    follow = joined[(joined['days_from_baseline'] >= FOLLOWUP_MIN_DAYS) & (joined['days_from_baseline'] <= FOLLOWUP_MAX_DAYS)].copy()
    if follow.empty:
        raise ValueError('No BMI follow-up rows found in the configured 2-year window.')

    follow['distance_to_target_days'] = (follow['days_from_baseline'] - FOLLOWUP_TARGET_DAYS).abs()
    follow = follow.sort_values(['participant_id', 'distance_to_target_days']).groupby('participant_id', as_index=False).first()
    target = follow[['participant_id', 'baseline_date', 'baseline_bmi', 'date', 'bmi', 'days_from_baseline']].copy()
    target = target.rename(columns={'date': 'followup_date', 'bmi': 'followup_bmi'})
    target['two_year_overweight_or_obesity'] = (target['followup_bmi'] >= BMI_CUTOFF).astype(int)
    target.to_csv(cache, index=False)
    print('Wrote target:', cache, target.shape)
    return target

target = build_two_year_obesity_target()
print('Target positive rate:')
display(target['two_year_overweight_or_obesity'].value_counts(dropna=False).rename('count').to_frame())
display(target[['participant_id', 'baseline_bmi', 'followup_bmi', 'days_from_baseline', 'two_year_overweight_or_obesity']].head())

## Build Age/Sex Covariates

In [ ]:
def load_covariates(participant_ids):
    cache = CACHE_DIR / 'two_year_obesity_covariates.csv'
    if cache.exists():
        cov = pd.read_csv(cache, low_memory=False)
        cov['participant_id'] = cov['participant_id'].astype(str)
        print('Loaded cached covariates:', cache, cov.shape)
        return cov

    specs = [
        ('anthropometrics', 'age_sex'),
        ('body_composition', 'age_sex'),
        ('blood_tests', 'age_sex'),
        ('population', 'population'),
    ]
    frames = []
    cov_rx = re.compile(r'^age$|age_at|sex$|gender$|year_of_birth', re.IGNORECASE)
    for dataset, table in specs:
        frame, _ = try_load_pheno_table(dataset, table)
        if frame is None:
            continue
        pid_col = find_participant_col(frame)
        if pid_col is None:
            continue
        matches = [c for c in frame.columns if cov_rx.search(str(c))]
        if not matches:
            continue
        part = frame[[pid_col] + matches].copy().rename(columns={pid_col: 'participant_id'})
        part['participant_id'] = part['participant_id'].astype(str)
        rename = {}
        for c in matches:
            lc = str(c).lower()
            if 'sex' in lc or 'gender' in lc:
                rename[c] = 'sex'
            elif 'year_of_birth' in lc:
                rename[c] = 'year_of_birth'
            elif 'age' in lc:
                rename[c] = 'age'
        part = part.rename(columns=rename)
        keep = ['participant_id'] + [c for c in ['age', 'sex', 'year_of_birth'] if c in part.columns]
        part = part[keep]
        if 'age' in part.columns:
            part['age'] = pd.to_numeric(part['age'], errors='coerce')
        if 'year_of_birth' in part.columns and 'age' not in part.columns:
            part['year_of_birth'] = pd.to_numeric(part['year_of_birth'], errors='coerce')
            part['age'] = 2022 - part['year_of_birth']
            part = part.drop(columns=['year_of_birth'])
        elif 'year_of_birth' in part.columns:
            part = part.drop(columns=['year_of_birth'])
        frames.append(part.groupby('participant_id', as_index=False).first())

    cov = pd.DataFrame({'participant_id': pd.Series(participant_ids).astype(str).unique()})
    for part in frames:
        for col in [c for c in part.columns if c != 'participant_id']:
            if col not in cov.columns:
                cov = cov.merge(part[['participant_id', col]], on='participant_id', how='left')
            else:
                add = part[['participant_id', col]].rename(columns={col: f'{col}_new'})
                cov = cov.merge(add, on='participant_id', how='left')
                cov[col] = cov[col].combine_first(cov[f'{col}_new'])
                cov = cov.drop(columns=[f'{col}_new'])
    cov.to_csv(cache, index=False)
    print('Wrote covariates:', cache, cov.shape)
    return cov

covariates = load_covariates(target['participant_id'])
print('Covariate non-null counts:')
display(covariates.notna().sum())
display(covariates.head())

## Build 800-kcal-Filtered Participant Diet Features

In [ ]:
PAPER_BASIC_NUTRIENT_NAMES = [
    'Energy', 'Protein', 'Total lipid (fat)', 'Carbohydrate, by difference',
    'Fiber, total dietary', 'Sodium, Na', 'Water', 'Alcohol, ethyl',
]
BASIC_NUTRIENT_KEYS = {clean_feature_name(x) for x in PAPER_BASIC_NUTRIENT_NAMES}
PAPER_BASIC_PATTERNS = {
    'energy': re.compile(r'(^|_)energy($|_)|calorie|kcal', re.IGNORECASE),
    'protein': re.compile(r'(^|_)protein($|_)', re.IGNORECASE),
    'total_lipid_fat': re.compile(r'total_lipid|lipid|total_fat|(^|_)fat($|_)', re.IGNORECASE),
    'carbohydrate_by_difference': re.compile(r'carbohydrate|(^|_)carb($|_)', re.IGNORECASE),
    'fiber_total_dietary': re.compile(r'fiber|fibre', re.IGNORECASE),
    'sodium_na': re.compile(r'sodium|(^|_)na($|_)', re.IGNORECASE),
    'water': re.compile(r'(^|_)water($|_)', re.IGNORECASE),
    'alcohol_ethyl': re.compile(r'alcohol|ethyl', re.IGNORECASE),
}


def paper_basic_nutrient_kind(col):
    name = clean_feature_name(col)
    if name in BASIC_NUTRIENT_KEYS:
        return name
    for kind, pattern in PAPER_BASIC_PATTERNS.items():
        if pattern.search(name):
            return kind
    return None


def load_diet_events():
    for path in [TRE_INPUTS / 'diet_logging_events.parquet', TRE_INPUTS / 'diet_logging_events.csv']:
        if path.exists():
            print('Loading diet events:', path)
            return read_any(path)
    frame, _ = try_load_pheno_table('diet_logging', 'diet_logging_events')
    if frame is None:
        raise FileNotFoundError('Could not load diet_logging_events.')
    TRE_INPUTS.mkdir(parents=True, exist_ok=True)
    out = TRE_INPUTS / 'diet_logging_events.csv'
    frame.to_csv(out, index=False)
    print('Wrote diet events:', out)
    return frame


def choose_day_col(df):
    if 'logging_day' in df.columns:
        return 'logging_day'
    for c in ['collection_date', 'local_date', 'date']:
        if c in df.columns:
            return c
    for c in ['collection_timestamp', 'local_timestamp', 'timestamp']:
        if c in df.columns:
            return c
    return None


def build_filtered_participant_food():
    cache = CACHE_DIR / f'diet_participant_food_gef_{int(MIN_DAILY_KCAL)}kcal.csv'
    days_cache = CACHE_DIR / f'diet_valid_days_gef_{int(MIN_DAILY_KCAL)}kcal.csv'
    if cache.exists() and days_cache.exists():
        diet = pd.read_csv(cache, low_memory=False)
        valid_days = pd.read_csv(days_cache, low_memory=False)
        diet['participant_id'] = diet['participant_id'].astype(str)
        valid_days['participant_id'] = valid_days['participant_id'].astype(str)
        print('Loaded cached filtered diet:', cache, diet.shape)
        return diet, valid_days

    events = load_diet_events()
    required = ['participant_id', 'food_id', 'weight_g']
    missing = [c for c in required if c not in events.columns]
    if missing:
        raise ValueError(f'Diet events missing required columns: {missing}')
    day_col = choose_day_col(events)
    if day_col is None:
        print('No day column found; aggregating all events without the 800 kcal/day filter.')
        events['_diet_day'] = 'all_days'
    else:
        events['_diet_day'] = events[day_col]
        if day_col not in ['logging_day']:
            events['_diet_day'] = pd.to_datetime(events['_diet_day'], errors='coerce').dt.date.astype(str)

    events['participant_id'] = events['participant_id'].astype(str)
    events['food_id'] = events['food_id'].astype(str)
    events['weight_g'] = pd.to_numeric(events['weight_g'], errors='coerce').fillna(0.0)

    kcal_col = find_first_matching_column(events, [r'calories_kcal', r'energy_kcal', r'kcal', r'calorie'])
    if kcal_col is not None:
        events['_kcal'] = pd.to_numeric(events[kcal_col], errors='coerce').fillna(0.0)
        day_energy = events.groupby(['participant_id', '_diet_day'], as_index=False)['_kcal'].sum()
        valid_day_keys = day_energy[day_energy['_kcal'] >= MIN_DAILY_KCAL][['participant_id', '_diet_day']]
        before_days = len(day_energy)
        events = events.merge(valid_day_keys, on=['participant_id', '_diet_day'], how='inner')
        print('Applied 800 kcal/day filter using', kcal_col, '| valid days:', len(valid_day_keys), 'of', before_days)
    else:
        print('No calorie column found; aggregating all days without the 800 kcal/day filter.')

    valid_days = events.groupby('participant_id', as_index=False)['_diet_day'].nunique().rename(columns={'_diet_day': 'valid_diet_days'})
    diet = events.groupby(['participant_id', 'food_id'], as_index=False)['weight_g'].sum()
    diet = diet.merge(valid_days, on='participant_id', how='left')
    diet.to_csv(cache, index=False)
    valid_days.to_csv(days_cache, index=False)
    print('Wrote filtered diet:', cache, diet.shape)
    return diet, valid_days

participant_food, valid_days = build_filtered_participant_food()
display(participant_food.head())
display(valid_days['valid_diet_days'].describe())

In [ ]:
def choose_ref_food_col(ref):
    for col in ['hpp_food_id', 'food_id']:
        if col in ref.columns:
            return col
    raise ValueError('Could not find hpp_food_id or food_id in feature table')


def feature_columns(ref, ref_food_col, feature_mode):
    if feature_mode == 'embedding':
        cols = [c for c in ref.columns if str(c).startswith('embedding_')]
    else:
        exclude = {ref_food_col, 'food_id', 'hpp_food_id', 'food_name', 'short_food_name', 'product_name'}
        cols = [c for c in ref.columns if c not in exclude and pd.api.types.is_numeric_dtype(ref[c])]
    return list(dict.fromkeys(cols))


def build_participant_x(fs, batch_size=None):
    if batch_size is None:
        batch_size = X_BUILD_BATCH_SIZE
    out = CACHE_DIR / f"X_{fs['name']}_participant_gef_{int(MIN_DAILY_KCAL)}kcal.csv"
    if out.exists():
        x = pd.read_csv(out, low_memory=False)
        x['participant_id'] = x['participant_id'].astype(str)
        print('Loaded cached X:', out, x.shape)
        return x

    print('Building X:', fs['name'], '| batch_size=', batch_size, flush=True)
    ref = read_any(PROJECT_ROOT / fs['path'])
    ref_food_col = choose_ref_food_col(ref)
    cols = feature_columns(ref, ref_food_col, fs['feature_mode'])
    if not cols:
        raise ValueError(f"No numeric feature columns found for {fs['name']}")
    ref = ref[[ref_food_col] + cols].copy()
    ref['_food_join_id'] = ref[ref_food_col].astype(str)

    diet = participant_food.copy()
    diet['_food_join_id'] = diet['food_id'].astype(str)
    total_grams = diet.groupby('participant_id')['weight_g'].sum().replace(0, np.nan)
    days = valid_days.set_index('participant_id')['valid_diet_days'].replace(0, np.nan)

    parts = []
    for start in range(0, len(cols), batch_size):
        batch = cols[start:start + batch_size]
        print('  feature batch', start + 1, '-', min(start + batch_size, len(cols)), 'of', len(cols), flush=True)
        merged = diet[['participant_id', '_food_join_id', 'weight_g']].merge(ref[['_food_join_id'] + batch], on='_food_join_id', how='left')
        values = merged[batch].apply(pd.to_numeric, errors='coerce').fillna(0.0)
        if fs['feature_mode'] == 'enriched':
            scaled = values.mul(merged['weight_g'].to_numpy() / 100.0, axis=0)
            agg = scaled.assign(participant_id=merged['participant_id'].values).groupby('participant_id').sum(numeric_only=True)
            agg = agg.div(days, axis=0).fillna(0.0)
            prefix = 'enriched_daily_'
        elif fs['feature_mode'] in ['kg', 'embedding']:
            scaled = values.mul(merged['weight_g'].to_numpy(), axis=0)
            agg = scaled.assign(participant_id=merged['participant_id'].values).groupby('participant_id').sum(numeric_only=True)
            agg = agg.div(total_grams, axis=0).fillna(0.0)
            prefix = 'kg_weighted_' if fs['feature_mode'] == 'kg' else 'food_card_weighted_'
        else:
            raise ValueError(fs['feature_mode'])
        agg.columns = [prefix + str(c) for c in agg.columns]
        parts.append(agg)
        del merged, values, scaled, agg
        gc.collect()
    x = pd.concat(parts, axis=1).reset_index()
    x.to_csv(out, index=False)
    print('Wrote X:', out, x.shape, flush=True)
    del ref, diet, total_grams, days, parts
    gc.collect()
    return x

x_tables = {}
for fs in FEATURE_SETS:
    x_tables[fs['name']] = build_participant_x(fs)
    gc.collect()
print('Built/loaded X tables:', {k: v.shape for k, v in x_tables.items()})

## Create Model Arms

In [ ]:
basic_x = x_tables['basic_nutrimatch']
all_basic_feature_cols = [c for c in basic_x.columns if c != 'participant_id' and pd.api.types.is_numeric_dtype(basic_x[c])]

paper_basic_cols_by_kind = {}
for col in all_basic_feature_cols:
    kind = paper_basic_nutrient_kind(col)
    if kind and kind not in paper_basic_cols_by_kind:
        paper_basic_cols_by_kind[kind] = col
paper_basic_cols = list(paper_basic_cols_by_kind.values())

print('Paper-basic nutrient kinds found:', sorted(paper_basic_cols_by_kind))
print('Paper-basic nutrient columns found:', paper_basic_cols)
if not paper_basic_cols:
    print('First 80 available NutriMatch columns:', all_basic_feature_cols[:80])
    raise ValueError('No paper-basic nutrient columns were found.')

covariate_cols = [c for c in ['age', 'sex'] if c in covariates.columns and covariates[c].notna().any()]
print('Using covariates:', covariate_cols)

arms = [
    {'arm': 'age_sex_only', 'label': ARM_LABELS['age_sex_only'], 'x': covariates[['participant_id'] + covariate_cols].copy()},
    {'arm': 'paper_basic_nutrients', 'label': ARM_LABELS['paper_basic_nutrients'], 'x': basic_x[['participant_id'] + paper_basic_cols].merge(covariates[['participant_id'] + covariate_cols], on='participant_id', how='left')},
    {'arm': 'nutrimatch_all', 'label': ARM_LABELS['nutrimatch_all'], 'x': basic_x.merge(covariates[['participant_id'] + covariate_cols], on='participant_id', how='left')},
]
for fs in FEATURE_SETS:
    if fs['name'] == 'basic_nutrimatch':
        continue
    x = x_tables[fs['name']].merge(covariates[['participant_id'] + covariate_cols], on='participant_id', how='left')
    arms.append({'arm': fs['name'], 'label': ARM_LABELS[fs['name']], 'x': x})

for arm in arms:
    print(arm['arm'], arm['x'].shape, arm['label'])

## Train And Evaluate AUROC

In [ ]:
def make_onehot():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def model_for(model_name=MODEL_NAME):
    if model_name == 'lightgbm':
        if LIGHTGBM_AVAILABLE:
            # Primary paper-aligned model: the paper states LightGBM with 5-fold CV.
            # The paper does not publish exact hyperparameters, so these conservative
            # values are fixed and recorded in the result table.
            return LGBMClassifier(
                n_estimators=300,
                learning_rate=0.03,
                random_state=RANDOM_STATE,
                verbose=-1,
            )
        if not ALLOW_MODEL_FALLBACK:
            raise ImportError('LightGBM is required for the paper-aligned primary model but is not installed.')
        print('LightGBM requested but unavailable; falling back to HistGradientBoostingClassifier.')
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.03, random_state=RANDOM_STATE)
    if model_name == 'hist_gradient_boosting':
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.03, random_state=RANDOM_STATE)
    if model_name == 'random_forest':
        return RandomForestClassifier(
            n_estimators=500,
            max_features='sqrt',
            class_weight='balanced_subsample',
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    raise ValueError(f'Unknown model_name: {model_name}')


def effective_model_name(model_name=MODEL_NAME):
    if model_name == 'lightgbm' and not LIGHTGBM_AVAILABLE and ALLOW_MODEL_FALLBACK:
        return 'hist_gradient_boosting_fallback'
    return model_name


def pipeline_for(X, model_name=MODEL_NAME):
    numeric_cols = X.select_dtypes(include=[np.number, bool]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in numeric_cols]
    # Paper-level preprocessing: no scaling for tree models; impute numeric/categorical
    # values and one-hot encode categorical covariates such as sex.
    pre = ColumnTransformer(
        transformers=[
            ('num', Pipeline([('impute', SimpleImputer())]), numeric_cols),
            ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', make_onehot())]), cat_cols),
        ],
        remainder='drop',
    )
    return Pipeline([('pre', pre), ('model', model_for(model_name))])


def evaluate_arm(arm, model_name=MODEL_NAME):
    y_table = target[['participant_id', 'two_year_overweight_or_obesity']].copy()
    y_table['participant_id'] = y_table['participant_id'].astype(str)
    merged = arm['x'].merge(y_table, on='participant_id', how='inner')
    participants = merged['participant_id'].copy()
    y = merged['two_year_overweight_or_obesity'].astype(int)
    X = merged.drop(columns=['participant_id', 'two_year_overweight_or_obesity'])
    counts = y.value_counts()
    if y.nunique() < 2 or counts.min() < N_SPLITS:
        raise ValueError(f"Not enough cases/controls for {arm['arm']}: {counts.to_dict()}")
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    estimator = pipeline_for(X, model_name=model_name)
    fold_rows = []
    oof_rows = []
    for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
        est = clone(estimator)
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        est.fit(X_train, y_train)
        pred = est.predict(X_test)
        if hasattr(est, 'predict_proba'):
            score = est.predict_proba(X_test)[:, 1]
        else:
            score = est.decision_function(X_test)
        fold_rows.append({
            'fold': fold,
            'auroc': float(roc_auc_score(y_test, score)),
            'auprc': float(average_precision_score(y_test, score)),
            'accuracy': float(accuracy_score(y_test, pred)),
            'balanced_accuracy': float(balanced_accuracy_score(y_test, pred)),
            'f1_macro': float(f1_score(y_test, pred, average='macro', zero_division=0)),
            'n_test': int(len(test_idx)),
        })
        for pid, yt, yp, sc in zip(participants.iloc[test_idx], y_test, pred, score):
            oof_rows.append({'participant_id': pid, 'fold': fold, 'y_true': int(yt), 'y_pred': int(yp), 'y_score': float(sc)})
    fold_df = pd.DataFrame(fold_rows)
    oof = pd.DataFrame(oof_rows)
    summary = {
        'arm': arm['arm'],
        'label': arm['label'],
        'model': effective_model_name(model_name),
        'n': int(len(y)),
        'positive_n': int(y.sum()),
        'positive_rate': float(y.mean()),
        'feature_count': int(X.shape[1]),
        'auroc_mean': float(fold_df['auroc'].mean()),
        'auroc_std': float(fold_df['auroc'].std()),
        'auprc_mean': float(fold_df['auprc'].mean()),
        'auprc_std': float(fold_df['auprc'].std()),
        'accuracy_mean': float(fold_df['accuracy'].mean()),
        'balanced_accuracy_mean': float(fold_df['balanced_accuracy'].mean()),
        'f1_macro_mean': float(fold_df['f1_macro'].mean()),
        'oof_auroc': float(roc_auc_score(oof['y_true'], oof['y_score'])),
        'oof_auprc': float(average_precision_score(oof['y_true'], oof['y_score'])),
    }
    oof['arm'] = arm['arm']
    oof['label'] = arm['label']
    return summary, fold_df, oof

results_path = OUT_DIR / 'two_year_obesity_model_results.csv'
fold_path = OUT_DIR / 'two_year_obesity_fold_metrics.csv'
oof_path = OUT_DIR / 'two_year_obesity_oof_predictions.csv'

if RUN_TRAINING:
    summaries = []
    folds = []
    oofs = []
    for arm in arms:
        print('Training:', arm['arm'], arm['label'])
        summary, fold_df, oof = evaluate_arm(arm, model_name=MODEL_NAME)
        summaries.append(summary)
        fold_df['arm'] = arm['arm']
        fold_df['label'] = arm['label']
        folds.append(fold_df)
        oofs.append(oof)
        print(' ', 'AUROC:', summary['auroc_mean'], '+/-', summary['auroc_std'], '| OOF:', summary['oof_auroc'])
    results = pd.DataFrame(summaries)
    fold_metrics = pd.concat(folds, ignore_index=True)
    oof_predictions = pd.concat(oofs, ignore_index=True)
    results.to_csv(results_path, index=False)
    fold_metrics.to_csv(fold_path, index=False)
    oof_predictions.to_csv(oof_path, index=False)
    print('Wrote:', results_path)
    print('Wrote:', fold_path)
    print('Wrote:', oof_path)
elif results_path.exists():
    results = pd.read_csv(results_path, low_memory=False)
    fold_metrics = pd.read_csv(fold_path, low_memory=False) if fold_path.exists() else pd.DataFrame()
    oof_predictions = pd.read_csv(oof_path, low_memory=False) if oof_path.exists() else pd.DataFrame()
    print('Loaded saved results:', results_path, results.shape)
else:
    results = pd.DataFrame()
    fold_metrics = pd.DataFrame()
    oof_predictions = pd.DataFrame()
    print('Training skipped and no saved results found. Run the background runner first.')

display(results.sort_values('auroc_mean', ascending=False) if not results.empty else results)

## Supplementary Random Forest Classification And Feature Importance

This section is supplementary, not the paper-primary model. It evaluates Random Forest only for the full NutriMatch arm and the best-performing enhanced alternative from the primary AUROC table, then compares feature importances side by side.


In [ ]:
rf_results_path = OUT_DIR / 'supplementary_random_forest_two_year_obesity_results.csv'
rf_fold_path = OUT_DIR / 'supplementary_random_forest_two_year_obesity_fold_metrics.csv'
rf_oof_path = OUT_DIR / 'supplementary_random_forest_two_year_obesity_oof_predictions.csv'
rf_importance_path = OUT_DIR / 'supplementary_random_forest_feature_importance.csv'


def feature_names_from_preprocessor(fitted_pipeline, original_X):
    pre = fitted_pipeline.named_steps['pre']
    try:
        return pre.get_feature_names_out().tolist()
    except Exception:
        names = []
        for name, transformer, cols in pre.transformers_:
            if name == 'remainder' or transformer == 'drop':
                continue
            if name == 'num':
                names.extend([f'num__{c}' for c in cols])
            elif name == 'cat':
                try:
                    onehot = transformer.named_steps['onehot']
                    names.extend(onehot.get_feature_names_out(cols).tolist())
                except Exception:
                    names.extend([f'cat__{c}' for c in cols])
        return names


def fit_random_forest_importance(arm):
    y_table = target[['participant_id', 'two_year_overweight_or_obesity']].copy()
    y_table['participant_id'] = y_table['participant_id'].astype(str)
    merged = arm['x'].merge(y_table, on='participant_id', how='inner')
    y = merged['two_year_overweight_or_obesity'].astype(int)
    X = merged.drop(columns=['participant_id', 'two_year_overweight_or_obesity'])
    est = pipeline_for(X, model_name='random_forest')
    est.fit(X, y)
    model = est.named_steps['model']
    names = feature_names_from_preprocessor(est, X)
    importances = getattr(model, 'feature_importances_', None)
    if importances is None:
        return pd.DataFrame()
    if len(names) != len(importances):
        names = [f'feature_{i}' for i in range(len(importances))]
    out = pd.DataFrame({
        'arm': arm['arm'],
        'label': arm['label'],
        'feature': names,
        'importance': importances,
    }).sort_values('importance', ascending=False)
    return out

if RUN_TRAINING and RUN_RF_SUPPLEMENT and not results.empty:
    candidate = results[~results['arm'].isin(['age_sex_only', 'paper_basic_nutrients', 'nutrimatch_all'])].copy()
    if candidate.empty:
        print('No enhanced alternative arms found for Random Forest supplement.')
        rf_results = pd.DataFrame()
        rf_fold_metrics = pd.DataFrame()
        rf_oof_predictions = pd.DataFrame()
        rf_importance = pd.DataFrame()
    else:
        best_alt_arm = candidate.sort_values('auroc_mean', ascending=False).iloc[0]['arm']
        rf_arm_names = ['nutrimatch_all', best_alt_arm]
        print('Random Forest supplement arms:', rf_arm_names)
        rf_summaries = []
        rf_folds = []
        rf_oofs = []
        rf_importances = []
        arm_lookup = {a['arm']: a for a in arms}
        for arm_name in rf_arm_names:
            arm = arm_lookup[arm_name]
            print('Training supplementary RF:', arm['arm'], arm['label'])
            summary, fold_df, oof = evaluate_arm(arm, model_name='random_forest')
            rf_summaries.append(summary)
            fold_df['arm'] = arm['arm']
            fold_df['label'] = arm['label']
            rf_folds.append(fold_df)
            rf_oofs.append(oof)
            rf_importances.append(fit_random_forest_importance(arm))
            print(' ', 'RF AUROC:', summary['auroc_mean'], '+/-', summary['auroc_std'], '| OOF:', summary['oof_auroc'])
        rf_results = pd.DataFrame(rf_summaries)
        rf_fold_metrics = pd.concat(rf_folds, ignore_index=True)
        rf_oof_predictions = pd.concat(rf_oofs, ignore_index=True)
        rf_importance = pd.concat(rf_importances, ignore_index=True) if rf_importances else pd.DataFrame()
        rf_results.to_csv(rf_results_path, index=False)
        rf_fold_metrics.to_csv(rf_fold_path, index=False)
        rf_oof_predictions.to_csv(rf_oof_path, index=False)
        rf_importance.to_csv(rf_importance_path, index=False)
        print('Wrote RF supplement:', rf_results_path)
elif rf_results_path.exists():
    rf_results = pd.read_csv(rf_results_path, low_memory=False)
    rf_fold_metrics = pd.read_csv(rf_fold_path, low_memory=False) if rf_fold_path.exists() else pd.DataFrame()
    rf_oof_predictions = pd.read_csv(rf_oof_path, low_memory=False) if rf_oof_path.exists() else pd.DataFrame()
    rf_importance = pd.read_csv(rf_importance_path, low_memory=False) if rf_importance_path.exists() else pd.DataFrame()
    print('Loaded saved RF supplement:', rf_results_path, rf_results.shape)
else:
    rf_results = pd.DataFrame()
    rf_fold_metrics = pd.DataFrame()
    rf_oof_predictions = pd.DataFrame()
    rf_importance = pd.DataFrame()
    print('Random Forest supplement skipped or no saved supplement found yet.')

display(rf_results.sort_values('auroc_mean', ascending=False) if not rf_results.empty else rf_results)

## Paper-Style Figure 3c ROC Plots

In [ ]:
def save_fig(path):
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=240, bbox_inches='tight')
    print('Wrote:', path)


def plot_roc(arms_to_plot, path, title):
    plt.figure(figsize=(7, 6))
    rows = []
    for arm in arms_to_plot:
        p = oof_predictions[oof_predictions['arm'].eq(arm)].copy()
        if p.empty:
            continue
        y_true = p['y_true'].astype(int)
        y_score = pd.to_numeric(p['y_score'], errors='coerce')
        keep = y_score.notna()
        y_true = y_true.loc[keep]
        y_score = y_score.loc[keep]
        fpr, tpr, _ = roc_curve(y_true, y_score)
        auc = roc_auc_score(y_true, y_score)
        fold_auc = results.loc[results['arm'].eq(arm), 'auroc_mean'].iloc[0]
        fold_sd = results.loc[results['arm'].eq(arm), 'auroc_std'].iloc[0]
        label = ARM_LABELS.get(arm, arm)
        plt.plot(fpr, tpr, lw=2, label=f'{label} (AUC={fold_auc:.3f}±{fold_sd:.3f})')
        rows.append({'arm': arm, 'label': label, 'oof_auroc': auc, 'fold_auroc_mean': fold_auc, 'fold_auroc_std': fold_sd})
    plt.plot([0, 1], [0, 1], '--', color='black', lw=1)
    plt.xlabel('False positive rate')
    plt.ylabel('True positive rate')
    plt.title(title)
    plt.legend(fontsize=8)
    save_fig(path)
    plt.show()
    return pd.DataFrame(rows)

if not results.empty and not oof_predictions.empty:
    paper_three = [a for a in ['age_sex_only', 'paper_basic_nutrients', 'nutrimatch_all'] if a in set(oof_predictions['arm'])]
    paper_three_auc = plot_roc(
        paper_three,
        FIG_DIR / 'fig3c_paper_three_two_year_overweight_obesity_roc.png',
        '2-year overweight/obesity prediction: paper feature sets',
    )
    paper_three_auc.to_csv(OUT_DIR / 'fig3c_paper_three_auc_table.csv', index=False)
    display(paper_three_auc)

    all_arms = [a for a in ARM_ORDER if a in set(oof_predictions['arm'])]
    all_auc = plot_roc(
        all_arms,
        FIG_DIR / 'fig3c_paper_plus_enhancements_two_year_overweight_obesity_roc.png',
        '2-year overweight/obesity prediction: NutriMatch plus enhancement arms',
    )
    all_auc.to_csv(OUT_DIR / 'fig3c_paper_plus_enhancements_auc_table.csv', index=False)
    display(all_auc.sort_values('fold_auroc_mean', ascending=False))
else:
    print('No OOF predictions loaded; run training first.')

## AUROC Summary Plot

In [ ]:
if not results.empty:
    plot_df = results.copy()
    plot_df['label'] = pd.Categorical(plot_df['label'], categories=[ARM_LABELS[a] for a in ARM_ORDER if a in ARM_LABELS], ordered=True)
    plot_df = plot_df.sort_values('auroc_mean', ascending=True)
    plt.figure(figsize=(10, max(5, 0.45 * len(plot_df))))
    plt.errorbar(plot_df['auroc_mean'], plot_df['label'], xerr=plot_df['auroc_std'], fmt='o', color='black', ecolor='gray', capsize=3)
    plt.axvline(0.5, color='black', lw=1, ls='--')
    plt.xlabel('Mean fold AUROC')
    plt.ylabel('Feature set')
    plt.title('2-year overweight/obesity AUROC by diet feature set')
    save_fig(FIG_DIR / 'two_year_overweight_obesity_auroc_by_feature_set.png')
    plt.show()

    base = results.set_index('arm')
    if 'paper_basic_nutrients' in base.index and 'nutrimatch_all' in base.index:
        print('Paper-style change: NutriMatch all nutrients vs basic nutrients =', float(base.loc['nutrimatch_all', 'auroc_mean'] - base.loc['paper_basic_nutrients', 'auroc_mean']))
    if 'nutrimatch_all' in base.index:
        enh = results[~results['arm'].isin(['age_sex_only', 'paper_basic_nutrients', 'nutrimatch_all'])].copy()
        enh['delta_auroc_vs_nutrimatch_all'] = enh['auroc_mean'] - float(base.loc['nutrimatch_all', 'auroc_mean'])
        enh = enh.sort_values('delta_auroc_vs_nutrimatch_all', ascending=False)
        enh.to_csv(OUT_DIR / 'enhancement_deltas_vs_nutrimatch_all.csv', index=False)
        display(enh[['arm', 'label', 'auroc_mean', 'auroc_std', 'delta_auroc_vs_nutrimatch_all', 'feature_count', 'n']])
else:
    print('No result table loaded.')

## Supplementary Random Forest Feature-Importance Plot


In [ ]:
if not rf_importance.empty:
    top_n = 25
    compare_arms = rf_importance['arm'].drop_duplicates().tolist()
    fig, axes = plt.subplots(1, len(compare_arms), figsize=(7 * len(compare_arms), 9), sharex=False)
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])
    for ax, arm_name in zip(axes, compare_arms):
        p = rf_importance[rf_importance['arm'].eq(arm_name)].sort_values('importance', ascending=False).head(top_n).copy()
        p['feature_short'] = p['feature'].str.replace(r'^(num|cat)__', '', regex=True).str.slice(0, 55)
        sns.barplot(data=p, y='feature_short', x='importance', ax=ax, color='#4C72B0')
        ax.set_title(ARM_LABELS.get(arm_name, arm_name))
        ax.set_xlabel('Random Forest impurity importance')
        ax.set_ylabel('')
    save_fig(FIG_DIR / 'supplementary_random_forest_feature_importance_compare.png')
    plt.show()

    pivot = rf_importance.pivot_table(index='feature', columns='arm', values='importance', aggfunc='sum').fillna(0)
    pivot.to_csv(OUT_DIR / 'supplementary_random_forest_feature_importance_wide.csv')
    display(rf_importance.groupby('arm').head(25))
else:
    print('No Random Forest feature importance table loaded.')

## Final Paper-Ready Table

In [ ]:
if not results.empty:
    table = results[[
        'arm', 'label', 'model', 'n', 'positive_n', 'positive_rate', 'feature_count',
        'auroc_mean', 'auroc_std', 'oof_auroc', 'auprc_mean', 'oof_auprc',
        'accuracy_mean', 'balanced_accuracy_mean', 'f1_macro_mean',
    ]].sort_values('auroc_mean', ascending=False)
    table.to_csv(OUT_DIR / 'paper_ready_two_year_obesity_table.csv', index=False)
    print('Wrote:', OUT_DIR / 'paper_ready_two_year_obesity_table.csv')
    display(table)
else:
    print('No result table loaded.')